# Lab 03 — Resource Accounting: Parameters, FLOPs, and Memory
**Goal:** Build a calculator for model size, rough training compute, optimizer memory, and throughput sensitivity.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
def bytes_gib(x): return x/(1024**3)
def model_weight_bytes(params, bytes_per_param=2):
    return params*bytes_per_param
for p in [1e8, 1e9, 7e9]:
    print(f"{p/1e9:.1f}B params -> BF16 weights {bytes_gib(model_weight_bytes(p)):.2f} GiB")


## Predict before running
Predict how much memory a 7B model needs if weights, gradients, and Adam states all exist in memory. Which term dominates?


In [ ]:
def rough_training_flops(params, tokens):
    # Common first-order dense LM planning heuristic: ~6 * N * D FLOPs.
    return 6*params*tokens
for n,d in [(1e8,2e9),(1e9,20e9),(7e9,140e9)]:
    print(f"N={n/1e9:.1f}B D={d/1e9:.1f}B -> {rough_training_flops(n,d)/1e18:.2f} EFLOPs")


## Explain the result
The rough calculation shows why parameter count alone is not a complete resource estimate. Optimizer state can dominate training memory; activations add another large, sequence-length-dependent term.


In [ ]:
def optimizer_memory(params, weight_bpp=2, grad_bpp=2, adam_state_bpp=8):
    return params*(weight_bpp+grad_bpp+adam_state_bpp)
for n in [1e8,1e9,7e9]:
    print(f"{n/1e9:.1f}B -> rough weights+grad+Adam states {bytes_gib(optimizer_memory(n)):.2f} GiB")
print("Important: activation memory and implementation overhead are not included.")


## Critical-thinking answers
These are planning approximations, not exact bills. Use them to compare orders of magnitude, then validate with a real profiler on the target stack. Scaling proposals should include assumptions and a safety margin.

### Sources
https://cs336.stanford.edu/


## Deepening: separate inference and training memory

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
def raw_weight_gib(params, bytes_per_parameter):
    return params * bytes_per_parameter / 2**30

for p in [0.6e9, 7e9, 27e9]:
    print(f"{p/1e9:g}B BF16 raw weights:", raw_weight_gib(p,2), "GiB")
    print(f"{p/1e9:g}B 4-bit raw weights:", raw_weight_gib(p,0.5), "GiB")

print("\nThese are weight-only estimates. Training adds gradients, optimizer, activations, temporary/runtime state.")


## Break experiment

Choose one model and write two separate budgets:

1. inference;
2. full fine-tuning.

Then list every term that appears in one but not the other.

### Report

weights → gradients → optimizer → activations → KV cache → runtime

Complete templates/EXPERIMENT_CARD.md for the final result.
